
# 01 — Product Discovery

## Retrieval Framework Validation

This notebook validates the shared Retrieval Interface using the local development dataset.

A lightweight lexical baseline is used to verify product retrieval, review retrieval, and metadata filtering independently of the final Vector Database.

The results provide an initial integration checkpoint before implementing BM25, Vector Search, and Hybrid Search.


In [1]:
from pathlib import Path

import pandas as pd

from digikala_assistant.retrieval.base import RetrievalBackend
from digikala_assistant.retrieval.mock import MockRetriever
from digikala_assistant.schemas import SearchFilters

cwd = Path.cwd().resolve()

project_root = next(
    path
    for path in [cwd, *cwd.parents]
    if (path / "pyproject.toml").exists()
)

sample_dir = project_root / "data" / "sample"

retriever = MockRetriever(
    products_path=sample_dir / "products_dev.csv",
    reviews_path=sample_dir / "reviews_dev.csv",
)

assert isinstance(retriever, RetrievalBackend)

products_sample = pd.read_csv(
    sample_dir / "products_dev.csv",
    usecols=["id", "title_fa"],
    nrows=30,
)

query = products_sample["title_fa"].dropna().iloc[0]

hits = retriever.search_products(query, top_k=5)

assert hits, "Product retrieval returned no results."

for hit in hits:
    print(
        hit.item.product_id,
        hit.item.title,
        hit.item.price,
        round(hit.score, 3),
    )

product_id = hits[0].item.product_id
reviews = retriever.get_product_reviews(product_id, limit=3)

assert reviews, "No reviews found for the selected product."

print(f"\nProduct ID: {product_id}")
print(f"Retrieved reviews: {len(reviews)}")

for review in reviews:
    print(f"[{review.review_id}] {review.body[:150]}")

review_hits = retriever.search_reviews(
    reviews[0].body,
    product_id=product_id,
    top_k=3,
)

assert review_hits, "Review retrieval returned no results."

print(f"\nReview search results: {len(review_hits)}")
print("Retrieval Framework validation passed.")


21342 ماساژور برقی بیورر مدل MG70 38000000.0 1.0
10849351 ماساژور برقی مدل پروانه ای کد ems 1050000.0 0.6
1803881 کمربند لاغری های-تک مدل HI-Bm114 51500000.0 0.6
9121617 ماساژور برقی اوآبس مدل KH-515 11850000.0 0.6
2383313 ماساژور برقی سایکل تیری مدل MG 05 61000000.0 0.6

Product ID: 21342
Retrieved reviews: 3
[1009924] حدود یکسال پیش تو پیشنهاد ویژه خریدمش ، و از اون موقع زیاد استفاده کردم و یه جورایی بهش معتاد شدم مخصوصا بعد از ورزش ، ماساژ خوبی ازش میگیرم و خستگی ا
[1011987] زمانیکه من خریدم دیجی کالا نداشت از بیرون تهیه کردم واگر از نظرات دوستان اگاه بودم این مدل را نمیخریدم وهایتک  بنظرم بعلت سبکی بهتر باشد
[1405095] این دو برند اصلا قابل مقایسه با هم نیستند
اگرچه (های تک) همین مدل رو کپی کرده اما وقتی این دو دستگاه رو در کنار هم قرار بدین و مقایسه کنید خواهید دید

Review search results: 3
Retrieval Framework validation passed.



## BM25 Retrieval and Metadata Filtering

This experiment introduces BM25 as a stronger lexical retrieval baseline.

The BM25 implementation is compared with the initial token-overlap approach using the same product query.

Metadata Filtering is also validated by restricting product prices. The experiment verifies retrieval functionality and filtering correctness, while a separate labeled evaluation set will be required to measure Ranking quality.


In [2]:
from digikala_assistant.retrieval.bm25 import BM25Retriever

bm25_retriever = BM25Retriever(
    products_path=sample_dir / "products_dev.csv",
    reviews_path=sample_dir / "reviews_dev.csv",
)

query = "ماساژور برقی"

baseline_hits = retriever.search_products(query, top_k=5)
bm25_hits = bm25_retriever.search_products(query, top_k=5)

print("LEXICAL BASELINE")
for hit in baseline_hits:
    print(f"{hit.score:.3f} | {hit.item.title}")

print("\nBM25 RESULTS")
for hit in bm25_hits:
    print(f"{hit.score:.3f} | {hit.item.title}")

filtered_hits = bm25_retriever.search_products(
    query,
    filters=SearchFilters(max_price=20_000_000),
    top_k=5,
)

assert filtered_hits
assert all(
    hit.item.price is not None
    and hit.item.price <= 20_000_000
    for hit in filtered_hits
)

print("\nFILTERED BM25 RESULTS")
for hit in filtered_hits:
    print(
        f"{hit.item.price:,.0f} | "
        f"{hit.score:.3f} | "
        f"{hit.item.title}"
    )

print("\nBM25 and Metadata Filtering validation passed.")


LEXICAL BASELINE
1.000 | ماساژور برقی مدل پروانه ای کد ems
1.000 | ماساژور برقی بیورر مدل MG70
1.000 | کمربند لاغری های-تک مدل HI-Bm114
1.000 | ماساژور برقی اوآبس مدل KH-515
1.000 | ماساژور برقی سایکل تیری مدل MG 05

BM25 RESULTS
22.147 | ماساژور برقی بیورر مدل MG70
21.733 | ماساژور برقی اوآبس مدل KH-515
21.334 | ماساژور برقی مدل پروانه ای کد ems
20.950 | ماساژور برقی سایکل تیری مدل MG 05
16.805 | کمربند لاغری های-تک مدل HI-Bm114

FILTERED BM25 RESULTS
11,850,000 | 21.733 | ماساژور برقی اوآبس مدل KH-515
1,050,000 | 21.334 | ماساژور برقی مدل پروانه ای کد ems
1,795,000 | 11.647 | ماساژور دستی مدل AL -117
2,070,000 | 10.598 | ماساژور دستی تن زیب مدل PBN-04 مجموعه 3 عددی
136,000 | 10.096 | بازوی رفع خارش بدن مدل E17

BM25 and Metadata Filtering validation passed.



## Brand, Category, and Review Retrieval Validation

This experiment extends the BM25 validation by testing brand and category filtering, along with product-specific review retrieval.

The objective is to verify that structured constraints are respected and that review search never returns evidence belonging to another product.

This is a functional validation step, not a quantitative evaluation of retrieval relevance.


In [3]:
selected_product = bm25_hits[0].item
product_id = selected_product.product_id

print(f"Selected product: {selected_product.title}")
print(f"Product ID: {product_id}")

if selected_product.brand:
    brand_hits = bm25_retriever.search_products(
        selected_product.title,
        filters=SearchFilters(brand=selected_product.brand),
        top_k=10,
    )

    assert brand_hits
    assert all(hit.item.brand is not None for hit in brand_hits)

    print(f"\nBrand filter: {selected_product.brand}")
    print(f"Matching products: {len(brand_hits)}")

if selected_product.category1:
    category_hits = bm25_retriever.search_products(
        selected_product.title,
        filters=SearchFilters(category=selected_product.category1),
        top_k=10,
    )

    assert category_hits

    print(f"\nCategory filter: {selected_product.category1}")
    print(f"Matching products: {len(category_hits)}")

print("\nBM25 REVIEW RETRIEVAL")

review_queries = ["ماساژ", "کیفیت", "قدرت"]

for review_query in review_queries:
    review_hits = bm25_retriever.search_reviews(
        review_query,
        product_id=product_id,
        top_k=3,
    )

    assert all(
        hit.item.product_id == product_id
        for hit in review_hits
    )

    print(f"\nQuery: {review_query}")
    print(f"Retrieved reviews: {len(review_hits)}")

    for hit in review_hits:
        print(
            f"[{hit.item.review_id}] "
            f"Score: {hit.score:.3f} | "
            f"{hit.item.body[:120]}"
        )

print("\nAdditional Retrieval validation completed.")


Selected product: ماساژور برقی بیورر مدل MG70
Product ID: 21342

Brand filter: بیورر
Matching products: 1

Category filter: ماساژور
Matching products: 9

BM25 REVIEW RETRIEVAL

Query: ماساژ
Retrieved reviews: 3
[35575243] Score: 2.547 | پیشنهاد میکنم حتما بخرید.
این دستگاه در فیزیوتراپی استفاده می شود.
حتما با ژل ماساژ انجام بدید. 
عالیهه
خیلی دوستش دارم
[50496364] Score: 2.438 | دوستان چون سبکه خودتون میتونید خودتون رو ماساژ بدید ولی اگه کیی رو دارید که اینکار رو براتون انجام میده مدل بالاتر بهتره
[2183681] Score: 2.160 | من اینو برای همسرم گرفتم. خیلی از خریدم راضیم
چون هم راحت میشه ازش استفاده کرد هم اینکه خوب ماساژ میده. برعکس خیلی از م

Query: کیفیت
Retrieved reviews: 3
[32138674] Score: 2.825 | کیفیت ساختش عالیه
[50694122] Score: 2.825 | کیفیت  خیلی خوب
[5066437] Score: 2.628 | بسیار با کیفیت و قوی.

Query: قدرت
Retrieved reviews: 3
[42335026] Score: 3.711 | عالی ، با قدرت ، کارآمد ، خیلی راضیم ازش
[41439382] Score: 3.190 | خیلی قدرت ضعیفی داره انگار اسباب بازیه 
پولاتونو انگار ر


## Embedding Model Connectivity and Semantic Similarity

Before building the Vector Search index, the project API is tested using an embedding model.

This experiment verifies:

- Authentication and connectivity to the embedding endpoint.
- Successful generation of numerical text embeddings.
- Consistent embedding dimensions.
- Basic semantic similarity using cosine similarity.

The experiment uses only a few short texts to minimize API usage. Full product indexing will begin after successful validation.


In [7]:
import os

import numpy as np
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(project_root / ".env", override=True)

api_key = os.getenv("QBC13_API_KEY")

if not api_key:
    raise RuntimeError("QBC13_API_KEY is not configured.")

client = OpenAI(
    api_key=api_key,
    base_url=os.getenv(
        "QBC13_API_BASE_URL",
        "https://193.93.169.12/v1",
    ),
    timeout=60.0,
    max_retries=2,
)

embedding_model = "openai/text-embedding-3-small"

texts = [
    "ماساژور مناسب برای رفع گرفتگی عضلات",
    "دستگاه ماساژ بدن برای کاهش درد عضلانی",
    "کتاب آموزش زبان انگلیسی",
]

response = client.embeddings.create(
    model=embedding_model,
    input=texts,
    encoding_format="float",
)

ordered_embeddings = sorted(
    response.data,
    key=lambda item: item.index,
)

vectors = np.asarray(
    [item.embedding for item in ordered_embeddings],
    dtype=np.float32,
)

assert len(vectors) == len(texts)
assert vectors.ndim == 2
assert np.isfinite(vectors).all()

norms = np.linalg.norm(vectors, axis=1, keepdims=True)
assert np.all(norms > 0)

normalized_vectors = vectors / norms
similarities = normalized_vectors @ normalized_vectors.T

print(f"Embedding model: {response.model}")
print(f"Number of embeddings: {len(vectors)}")
print(f"Embedding dimensions: {vectors.shape[1]}")

print("\nCOSINE SIMILARITY")
print(f"Related texts: {similarities[0, 1]:.4f}")
print(f"Unrelated texts: {similarities[0, 2]:.4f}")

if response.usage:
    print(f"\nTotal tokens: {response.usage.total_tokens}")

print("\nEmbedding API validation passed.")


Embedding model: openai/text-embedding-3-small
Number of embeddings: 3
Embedding dimensions: 1536

COSINE SIMILARITY
Related texts: 0.5319
Unrelated texts: 0.1463

Total tokens: 73

Embedding API validation passed.



## Building a Persistent Product Embedding Index

This stage generates semantic embeddings for the development product catalog using product titles, brands, and categories.

The indexing pipeline uses batch processing to control API usage and stores intermediate checkpoints to avoid repeating completed requests after interruptions.

The final index contains normalized embedding vectors aligned with product IDs, enabling efficient cosine-similarity search without regenerating product embeddings for every query.

This index is intended for local development and will later be integrated with the shared Vector Database through the Retrieval Interface.


In [8]:
import hashlib
import json

products_df = pd.read_csv(
    sample_dir / "products_dev.csv",
    low_memory=False,
).sort_values("id").reset_index(drop=True)

metadata_columns = [
    "title_fa",
    "Brand",
    "Category1",
    "Category2",
    "sub_category",
]

products_df[metadata_columns] = (
    products_df[metadata_columns].fillna("").astype(str)
)

product_ids = products_df["id"].to_numpy(dtype=np.int64)

documents = [
    " | ".join(
        f"{column}: {value}"
        for column, value in zip(metadata_columns, row)
        if value.strip()
    )
    for row in products_df[metadata_columns].itertuples(
        index=False,
        name=None,
    )
]

fingerprint = hashlib.sha256(
    json.dumps(
        {
            "model": embedding_model,
            "products": list(zip(product_ids.tolist(), documents)),
        },
        ensure_ascii=False,
    ).encode("utf-8")
).hexdigest()

index_dir = project_root / "data" / "vector_index"
checkpoint_dir = index_dir / f"product_batches_{fingerprint[:16]}"

checkpoint_dir.mkdir(parents=True, exist_ok=True)

batch_size = 64
embedding_batches = []
total_tokens = 0

for start in range(0, len(documents), batch_size):
    end = min(start + batch_size, len(documents))
    batch_number = start // batch_size
    batch_path = checkpoint_dir / f"batch_{batch_number:04d}.npz"

    expected_ids = product_ids[start:end]

    if batch_path.exists():
        with np.load(batch_path) as cached:
            batch_ids = cached["product_ids"]
            batch_vectors = cached["embeddings"]

        if not np.array_equal(batch_ids, expected_ids):
            raise ValueError(f"Invalid checkpoint: {batch_path.name}")

        if batch_vectors.ndim != 2 or len(batch_vectors) != len(batch_ids):
            raise ValueError(f"Invalid embedding dimensions: {batch_path.name}")

    else:
        response = client.embeddings.create(
            model=embedding_model,
            input=documents[start:end],
            encoding_format="float",
        )

        ordered = sorted(response.data, key=lambda item: item.index)

        if [item.index for item in ordered] != list(range(end - start)):
            raise ValueError("Embedding response indices are inconsistent.")

        batch_vectors = np.asarray(
            [item.embedding for item in ordered],
            dtype=np.float32,
        )

        if not np.isfinite(batch_vectors).all():
            raise ValueError("Non-finite embedding values detected.")

        norms = np.linalg.norm(batch_vectors, axis=1, keepdims=True)

        if np.any(norms == 0):
            raise ValueError("Zero-length embedding detected.")

        batch_vectors = batch_vectors / norms

        np.savez_compressed(
            batch_path,
            product_ids=expected_ids,
            embeddings=batch_vectors,
        )

        if response.usage is not None:
            total_tokens += response.usage.total_tokens

    embedding_batches.append(batch_vectors)

    print(
        f"Batch {batch_number + 1} | "
        f"Products {end:,}/{len(documents):,}",
        flush=True,
    )

product_embeddings = np.vstack(embedding_batches)

assert len(product_ids) == len(product_embeddings)
assert np.isfinite(product_embeddings).all()

index_path = index_dir / "products_embeddings.npz"

np.savez_compressed(
    index_path,
    product_ids=product_ids,
    embeddings=product_embeddings,
    model=embedding_model,
    fingerprint=fingerprint,
)

print(f"\nIndexed products: {len(product_ids):,}")
print(f"Embedding dimensions: {product_embeddings.shape[1]}")
print(f"New API tokens used: {total_tokens:,}")
print(f"Index size: {index_path.stat().st_size / 1024**2:.2f} MB")
print(f"Saved index: {index_path}")


Batch 1 | Products 64/5,060
Batch 2 | Products 128/5,060
Batch 3 | Products 192/5,060
Batch 4 | Products 256/5,060
Batch 5 | Products 320/5,060
Batch 6 | Products 384/5,060
Batch 7 | Products 448/5,060
Batch 8 | Products 512/5,060
Batch 9 | Products 576/5,060
Batch 10 | Products 640/5,060
Batch 11 | Products 704/5,060
Batch 12 | Products 768/5,060
Batch 13 | Products 832/5,060
Batch 14 | Products 896/5,060
Batch 15 | Products 960/5,060
Batch 16 | Products 1,024/5,060
Batch 17 | Products 1,088/5,060
Batch 18 | Products 1,152/5,060
Batch 19 | Products 1,216/5,060
Batch 20 | Products 1,280/5,060
Batch 21 | Products 1,344/5,060
Batch 22 | Products 1,408/5,060
Batch 23 | Products 1,472/5,060
Batch 24 | Products 1,536/5,060
Batch 25 | Products 1,600/5,060
Batch 26 | Products 1,664/5,060
Batch 27 | Products 1,728/5,060
Batch 28 | Products 1,792/5,060
Batch 29 | Products 1,856/5,060
Batch 30 | Products 1,920/5,060
Batch 31 | Products 1,984/5,060
Batch 32 | Products 2,048/5,060
Batch 33 | Produ


## Vector Search and Hybrid Retrieval Validation

This experiment loads the previously generated product embedding index and performs semantic retrieval using cosine similarity.

The VectorRetriever is combined with BM25Retriever through Reciprocal Rank Fusion (RRF). The experiment compares the top-ranked products returned by BM25, Vector Search, and Hybrid Search for the same query.

Metadata Filtering and compatibility with the shared Retrieval Interface are also verified. These tests validate system functionality; a labeled benchmark is still required to establish quantitative improvements in retrieval relevance.


In [9]:
from digikala_assistant.retrieval.vector import VectorRetriever
from digikala_assistant.retrieval.hybrid import HybridRetriever

vector_retriever = VectorRetriever(
    backend=bm25_retriever,
    index_path=index_path,
    client=client,
    embedding_model=embedding_model,
)

hybrid_retriever = HybridRetriever(
    lexical=bm25_retriever,
    semantic=vector_retriever,
)

assert isinstance(vector_retriever, RetrievalBackend)
assert isinstance(hybrid_retriever, RetrievalBackend)

query = "ماساژور برقی برای رفع گرفتگی عضلات"

retrievers = {
    "BM25": bm25_retriever,
    "VECTOR SEARCH": vector_retriever,
    "HYBRID SEARCH": hybrid_retriever,
}

for name, backend in retrievers.items():
    hits = backend.search_products(query, top_k=5)

    assert hits

    print(f"\n{name}")
    print("-" * 60)

    for rank, hit in enumerate(hits, start=1):
        print(
            f"{rank}. "
            f"ID: {hit.item.product_id} | "
            f"Score: {hit.score:.4f} | "
            f"{hit.item.title}"
        )

filtered_hits = hybrid_retriever.search_products(
    query,
    filters=SearchFilters(max_price=20_000_000),
    top_k=5,
)

assert filtered_hits
assert all(
    hit.item.price is not None
    and hit.item.price <= 20_000_000
    for hit in filtered_hits
)

assert len({
    hit.item.product_id for hit in filtered_hits
}) == len(filtered_hits)

print("\nFILTERED HYBRID RESULTS")

for hit in filtered_hits:
    print(
        f"{hit.item.price:,.0f} | "
        f"{hit.item.title}"
    )

print(f"\nNew query embedding tokens: {vector_retriever.embedding_tokens:,}")
print("Vector Search and Hybrid Retrieval validation passed.")



BM25
------------------------------------------------------------
1. ID: 21342 | Score: 22.1469 | ماساژور برقی بیورر مدل MG70
2. ID: 9121617 | Score: 21.7327 | ماساژور برقی اوآبس مدل KH-515
3. ID: 10849351 | Score: 21.3340 | ماساژور برقی مدل پروانه ای کد ems
4. ID: 2383313 | Score: 20.9500 | ماساژور برقی سایکل تیری مدل MG 05
5. ID: 2889969 | Score: 18.1872 | بازوی رفع خارش بدن مدل E17

VECTOR SEARCH
------------------------------------------------------------
1. ID: 2889969 | Score: 0.4163 | بازوی رفع خارش بدن مدل E17
2. ID: 21342 | Score: 0.4039 | ماساژور برقی بیورر مدل MG70
3. ID: 9121617 | Score: 0.3891 | ماساژور برقی اوآبس مدل KH-515
4. ID: 10849351 | Score: 0.3884 | ماساژور برقی مدل پروانه ای کد ems
5. ID: 2383313 | Score: 0.3637 | ماساژور برقی سایکل تیری مدل MG 05

HYBRID SEARCH
------------------------------------------------------------
1. ID: 21342 | Score: 0.0325 | ماساژور برقی بیورر مدل MG70
2. ID: 9121617 | Score: 0.0320 | ماساژور برقی اوآبس مدل KH-515
3. ID: 2889969 | Sco


## Initial Retrieval Quality Evaluation

This experiment compares BM25, Vector Search, and Hybrid Search using a small manually annotated relevance set.

The evaluation measures Precision@3 and NDCG@5 to examine whether relevant products appear near the top of the search results.

Relevance judgments are preliminary and based on product titles. This experiment is a diagnostic sanity check, not a representative benchmark. A larger evaluation set with multiple queries and human-validated judgments will be needed for reliable conclusions.


In [10]:
import math

relevance = {
    21342: 1,
    9121617: 1,
    10849351: 1,
    2383313: 1,
    2889969: 0,
}

evaluation_query = "ماساژور برقی برای رفع گرفتگی عضلات"


def precision_at_k(hits, judgments, k):
    selected = hits[:k]
    return sum(
        judgments[hit.item.product_id] > 0
        for hit in selected
    ) / k


def ndcg_at_k(hits, judgments, k):
    dcg = sum(
        judgments[hit.item.product_id] / math.log2(rank + 2)
        for rank, hit in enumerate(hits[:k])
    )

    ideal_labels = sorted(
        judgments.values(),
        reverse=True,
    )[:k]

    idcg = sum(
        label / math.log2(rank + 2)
        for rank, label in enumerate(ideal_labels)
    )

    return dcg / idcg if idcg else 0.0


evaluation_results = []

for name, backend in retrievers.items():
    hits = backend.search_products(
        evaluation_query,
        top_k=5,
    )

    unknown_ids = {
        hit.item.product_id
        for hit in hits
        if hit.item.product_id not in relevance
    }

    if unknown_ids:
        raise ValueError(
            f"Missing relevance judgments for {name}: {unknown_ids}"
        )

    evaluation_results.append(
        {
            "retriever": name,
            "precision@3": precision_at_k(hits, relevance, 3),
            "ndcg@5": ndcg_at_k(hits, relevance, 5),
        }
    )

evaluation_df = pd.DataFrame(evaluation_results)

display(evaluation_df.round(4))


,retriever,precision@3,ndcg@5
0,BM25,1.0000,1.0000
1,VECTOR SEARCH,0.6667,0.7606
2,HYBRID SEARCH,0.6667,0.9558



## Product Discovery Service and Recommendation Statistics

This experiment integrates the Hybrid Retrieval Backend with a Product Discovery Service.

The service enriches retrieved products with recommendation statistics computed from real customer reviews. It also supports filtering candidates by a minimum recommendation rate and a minimum number of labeled reviews.

This stage validates structured recommendation results. Automatic query constraint extraction and natural-language answer generation will be implemented separately.


In [11]:
from digikala_assistant.discovery.service import ProductDiscoveryService

discovery = ProductDiscoveryService(hybrid_retriever)

query = "ماساژور برقی برای رفع گرفتگی عضلات"

suggestions = discovery.search(
    query,
    top_k=5,
)

assert suggestions
assert len({s.product.product_id for s in suggestions}) == len(suggestions)

rows = []

for suggestion in suggestions:
    rows.append(
        {
            "product_id": suggestion.product.product_id,
            "title": suggestion.product.title,
            "price": suggestion.product.price,
            "reviews": suggestion.review_count,
            "recommended": suggestion.recommended_count,
            "not_recommended": suggestion.not_recommended_count,
            "recommendation_rate": suggestion.recommendation_rate,
            "retrieval_score": suggestion.retrieval_score,
        }
    )

results_df = pd.DataFrame(rows)

display(results_df.round(4))

satisfied_suggestions = discovery.search(
    query,
    top_k=5,
    min_recommendation_rate=0.70,
    min_labeled_reviews=10,
)

assert all(
    s.labeled_count >= 10
    and s.recommendation_rate is not None
    and s.recommendation_rate >= 0.70
    for s in satisfied_suggestions
)

print(f"Initial recommendations: {len(suggestions)}")
print(f"Recommendations meeting satisfaction criteria: {len(satisfied_suggestions)}")
print("Product Discovery Service validation passed.")


,product_id,title,price,reviews,recommended,not_recommended,recommendation_rate,retrieval_score
0,21342,ماساژور برقی بیورر مدل MG70,38000000.0,97,66,2,0.9706,0.0325
1,9121617,ماساژور برقی اوآبس مدل KH-515,11850000.0,41,17,4,0.8095,0.0320
2,2889969,بازوی رفع خارش بدن مدل E17,136000.0,28,20,0,1.0000,0.0318
3,10849351,ماساژور برقی مدل پروانه ای کد ems,1050000.0,200,94,66,0.5875,0.0315
4,2383313,ماساژور برقی سایکل تیری مدل MG 05,61000000.0,18,10,3,0.7692,0.0310


Initial recommendations: 5
Recommendations meeting satisfaction criteria: 5
Product Discovery Service validation passed.



## Natural-Language Query Understanding

This experiment introduces a rule-based Query Understanding component for Persian product requests.

The parser extracts brand, category, and customer satisfaction constraints from natural-language queries and converts them into structured search parameters.

The resulting QueryPlan is passed to ProductDiscoveryService, allowing the retrieval pipeline to apply explicit metadata and satisfaction filters.

Price conversion is intentionally disabled until the unit of the dataset's Price column is verified.


In [12]:
from digikala_assistant.discovery.query import QueryUnderstanding

brands = products_df["Brand"].dropna().unique().tolist()

categories = pd.concat(
    [
        products_df["Category1"],
        products_df["Category2"],
        products_df["sub_category"],
    ],
    ignore_index=True,
).dropna().unique().tolist()

query_parser = QueryUnderstanding(
    brands=brands,
    categories=categories,
)

user_query = "ماساژور برقی بیورر با رضایت بالا"

plan = query_parser.parse(user_query)

print("ORIGINAL QUERY")
print(user_query)

print("\nQUERY PLAN")
print(plan)

assert plan.filters.brand == "بیورر"
assert plan.min_recommendation_rate == 0.70

suggestions = discovery.search(
    plan.search_query,
    filters=plan.filters,
    top_k=5,
    min_recommendation_rate=plan.min_recommendation_rate,
    min_labeled_reviews=plan.min_labeled_reviews,
)

assert suggestions

assert all(
    suggestion.product.brand is not None
    and "بیورر" in suggestion.product.brand
    and suggestion.recommendation_rate is not None
    and suggestion.recommendation_rate >= 0.70
    for suggestion in suggestions
)

result_rows = [
    {
        "product_id": s.product.product_id,
        "title": s.product.title,
        "brand": s.product.brand,
        "price": s.product.price,
        "labeled_reviews": s.labeled_count,
        "recommendation_rate": s.recommendation_rate,
    }
    for s in suggestions
]

display(pd.DataFrame(result_rows).round(4))

print("\nQuery Understanding validation passed.")


ORIGINAL QUERY
ماساژور برقی بیورر با رضایت بالا

QUERY PLAN
QueryPlan(search_query='ماساژور برقی با', filters=SearchFilters(min_price=None, max_price=None, brand='بیورر', category='ماساژور برقی'), min_recommendation_rate=0.7, min_labeled_reviews=10)


,product_id,title,brand,price,labeled_reviews,recommendation_rate
0,21342,ماساژور برقی بیورر مدل MG70,بیورر,38000000.0,68,0.9706



Query Understanding validation passed.



## Grounded Product Recommendation Generation

This experiment completes the initial Product Discovery pipeline by combining natural-language Query Understanding, Hybrid Retrieval, structured recommendation statistics, and LLM-based answer generation.

The LLM receives product metadata, recommendation statistics, and selected real customer reviews as evidence.

The generated response is checked to ensure that cited review IDs belong to the supplied evidence. This provides an initial citation-integrity check, while factual grounding and answer quality require further evaluation.


In [13]:
import importlib

import digikala_assistant.discovery.query as query_module

from digikala_assistant.discovery.answer import ProductAnswerGenerator

importlib.reload(query_module)

query_parser = query_module.QueryUnderstanding(
    brands=brands,
    categories=categories,
)

user_query = "ماساژور برقی بیورر با رضایت بالا"
plan = query_parser.parse(user_query)

assert plan.search_query == "ماساژور برقی"

suggestions = discovery.search(
    plan.search_query,
    filters=plan.filters,
    top_k=3,
    min_recommendation_rate=plan.min_recommendation_rate,
    min_labeled_reviews=plan.min_labeled_reviews,
)

answer_generator = ProductAnswerGenerator(
    backend=hybrid_retriever,
    client=client,
)

result = answer_generator.generate(
    query=user_query,
    suggestions=suggestions,
)

assert result.product_ids
assert result.cited_review_ids

assert set(result.cited_review_ids).issubset(
    set(result.available_review_ids)
)

print("GENERATED ANSWER")
print("-" * 60)
print(result.answer)

print("\nVALIDATION")
print(f"Recommended products: {len(result.product_ids)}")
print(f"Available evidence IDs: {len(result.available_review_ids)}")
print(f"Cited Review IDs: {result.cited_review_ids}")
print(f"LLM tokens: {result.total_tokens}")

print("\nProduct Discovery Answer Generation validation passed.")


GENERATED ANSWER
------------------------------------------------------------
**ماساژور برقی بیورر مدل MG70**

این محصول با نرخ رضایت 97% (66 توصیه شده از 68 بررسی) نشان‌دهنده تجربه مثبت کاربران است. یکی از کاربران [review_id:1849854] اشاره کرده که این ماساژور برای مادرش هدیه‌ای مناسب بوده و او از راحتی و کارایی آن راضی است. همچنین، کاربر دیگری [review_id:1009924] از دوام و کارایی آن پس از یک سال استفاده رضایت دارد. با این حال، یک کاربر دیگر [review_id:39789355] به مشکلاتی در کیفیت و صدا اشاره کرده است. به طور کلی، این ماساژور به عنوان یک گزینه خوب برای خرید پیشنهاد می‌شود.

VALIDATION
Recommended products: 1
Available evidence IDs: 4
Cited Review IDs: (1009924, 1849854, 39789355)
LLM tokens: 824

Product Discovery Answer Generation validation passed.
